# Sizing the vcm_gen Pump

Adapted from `macros/sw_tgate_cmfb2/scripts/sizing/size_sw_tgate_cmfb2.ipynb`, which in turn follows `macros/ota_fcasc_fd/scripts/sizing/lookup_commands.ipynb`. The load it has to hold up is the C1 network of `sc_int2`, sized in `macros/sc_int2/scripts/sizing/size_sc_int2.ipynb`.

**Copyright 2026 Michael Koefinger**

SPDX-License-Identifier: Apache-2.0 WITH SHL-2.1

`vcm_gen` was sized by hand and by simulation. This notebook derives the sizes from the spec, then reads the drawn sizes out of the schematic and compares them. It changes nothing. Where the two disagree the table says so.

Every number here is a model estimate at tt, 300 K. Simulated values come from `macros/vcm_gen/README.md` and are sanity checks of one or a few runs, not a characterisation.

No transistor in this block carries a bias current, so there is no gm/ID step. The flying capacitor comes from the output resistance of a 2:1 pump, the reservoir from the ripple the `sc_int2` right plates cause, the switches from Ron against the phase and from the leakage of the off device. The capacitor values come from the `cap_cmomf` area law, the gate capacitance of the reservoir from the README.

In [1]:
# Packages
from pygmid import Lookup as lk
import numpy as np
import re
from pathlib import Path

In [2]:
# Spec
VDD     = 1.5
FS      = 1.024e6               # pump clock, the two phases of clk_noverlap
TS2     = 1 / (2 * FS)          # one phase
VCM     = VDD / 2
N_SETTLE = 8                    # time constants in a phase, as in the integrator notebook
C1_REAL = 64e-15                # sc_int2 C1 per branch, estimated real, README of sc_int2
C1_MODEL = 87.95e-15            # the same, model value the schematic of sc_int2 carries now
C1_BENCH = 63.56e-15            # what the vcm_gen benches used (two cap_cmomi units of an older sc_int2)
V_VG    = 25e-3                 # virtual ground below vcm in the closed loop, README of vcm_gen
V_ERR   = 5e-3                  # assumed allowed vcm error from that load, the README has no number
V_RIPPLE = 50e-3                # assumed allowed ripple of vcm at the sampling instant, peak to peak
CAP_DENS = 31.77e-15 / (4.2 * 12.4)   # F/um2, cap_cmomf on Metal3 to Metal4 from the README value of the unit
GATE_DENS = 3.8e-15             # F/um2, thick-oxide NMOS gate at 0.75 V, README (DC and AC ops)
CELL    = 'vcm_gen'
DATA    = Path('../../../macros/inverter/scripts/sizing/data')
SCH     = Path('../schematic') / (CELL + '.sch')

### Read the sizes out of the schematic

The schematic is the source of truth. Widths and lengths come out in um whichever way they are written, the flying capacitors are two vectors of four units.

In [3]:
def um(v):
    return float(v[:-1]) if v.endswith('u') else float(v) * 1e6

def read_sizes(path):
    txt = path.read_text()
    dev = {}
    for m in re.finditer(r'C \{(\w+)\.sym\} \S+ \S+ \S+ \S+ \{name=(\w+)(?:\[(\d+):(\d+)\])?\n(.*?)\n?\}', txt, re.S):
        sym, name, hi, lo, body = m.groups()
        if not (sym.startswith('sg13_') or sym == 'cap_cmomf'):
            continue
        a = dict(re.findall(r'^(\w+)=(\S+)$', body, re.M))
        n = 1 if hi is None else abs(int(hi) - int(lo)) + 1
        dev[name] = dict(sym=sym, w=um(a['w']), l=um(a['l']), m=int(a['m']) * n)
    return dev

DEV = read_sizes(SCH)
for k, d in DEV.items():
    print(f"{k:5s} {d['sym']:12s} {d['w']:5.1f}u / {d['l']:5.2f}u  m = {d['m']}")

MS1   sg13_hv_pmos   4.0u /  0.45u  m = 1
MS4   sg13_hv_nmos   2.0u /  0.45u  m = 1
MS2   sg13_lv_pmos   4.0u /  0.50u  m = 1
MS3   sg13_lv_nmos   4.0u /  0.50u  m = 1
CF1   cap_cmomf      4.2u / 12.40u  m = 4
CF2   cap_cmomf      4.2u / 12.40u  m = 4
CR    sg13_hv_nmos  10.0u / 10.00u  m = 7
MDEC  sg13_hv_nmos   4.0u / 10.00u  m = 2
MDR   sg13_hv_nmos  10.0u /  0.45u  m = 2
MDS   sg13_hv_nmos   1.5u /  0.45u  m = 1


### Flying capacitor from the output resistance

Per cycle the 2:1 pump moves charge `C * vcm` out of the reservoir in phase 1 and back in phase 2, which is an output resistance of `1 / (4 fs C)` with `C = CF1 + CF2`, since both sit between the same two nets. `sc_int2` loads `vcm` with `2 * C1 * fs` per volt, 130 nA/V with the bench C1 and 131 nA/V with the 64 fF real one. The loop holds the virtual grounds about 25 mV below `vcm`, so the load is a few nA and `Rout` turns it into a `vcm` error. The README has no number for that error. 5 mV is assumed here, which is under 1 % of the 0.75 V full scale.

The formula is checked against the exact two-phase charge sharing: phase 1 puts `CF` in series with the reservoir across `vdd`, phase 2 puts it in parallel. The per-cycle contraction of the error gives the time constant `-Ts / ln(lambda)` and with it the effective resistance `tau / CR`, and the load can be added to the same recursion.

In [4]:
G_LOAD = 2 * C1_REAL * FS
I_LOAD = G_LOAD * V_VG
R_OUT_MAX = V_ERR / I_LOAD
C_MIN = 1 / (4 * FS * R_OUT_MAX)
print(f'load {G_LOAD*1e9:.0f} nA/V, {I_LOAD*1e9:.1f} nA at {V_VG*1e3:.0f} mV, error {V_ERR*1e3:.0f} mV allows Rout <= {R_OUT_MAX/1e6:.2f} MOhm')
print(f'C = CF1 + CF2 >= 1 / (4 fs Rout) = {C_MIN*1e15:.0f} fF')

c_unit = CAP_DENS * DEV['CF1']['w'] * DEV['CF1']['l']
C_F = (DEV['CF1']['m'] + DEV['CF2']['m']) * c_unit
CR = DEV['CR']['m'] * DEV['CR']['w'] * DEV['CR']['l'] * GATE_DENS
R_OUT = 1 / (4 * FS * C_F)
print(f'drawn: {DEV["CF1"]["m"] + DEV["CF2"]["m"]} units of {c_unit*1e15:.2f} fF = {C_F*1e15:.0f} fF, Rout = {R_OUT/1e6:.2f} MOhm, error {R_OUT*I_LOAD*1e3:.1f} mV')
print(f'reservoir {DEV["CR"]["m"]} x {DEV["CR"]["w"]:.0f}u x {DEV["CR"]["l"]:.0f}u = {CR*1e12:.2f} pF at {GATE_DENS*1e15:.1f} fF/um2')

load 131 nA/V, 3.3 nA at 25 mV, error 5 mV allows Rout <= 1.53 MOhm
C = CF1 + CF2 >= 1 / (4 fs Rout) = 160 fF
drawn: 8 units of 31.77 fF = 254 fF, Rout = 0.96 MOhm, error 3.1 mV
reservoir 7 x 10u x 10u = 2.66 pF at 3.8 fF/um2


In [5]:
def cycle(v, c, cr, cl=0.0, vg=VCM, vdd=VDD):
    # one clock of the pump from vcm = v, returns vcm at the end of phase 1 and of phase 2
    v1 = (c * (vdd - v) + cr * v + cl * vg) / (c + cr + cl)       # CF in series with CR (and the load plates on vcm)
    v2 = (c * (vdd - v1) + cr * v1) / (c + cr)                    # CF in parallel with CR
    return v1, v2

# small-signal contraction per cycle gives the effective output resistance
lam = (cycle(VCM + 1e-3, C_F, CR)[1] - VCM) / 1e-3
r_eff = -1 / FS / (CR * np.log(lam))
print(f'contraction per clock {lam:.4f}, Rout from the recursion {r_eff/1e6:.2f} MOhm, 1/(4 fs C) = {R_OUT/1e6:.2f} MOhm')

# start-up from 0 V, clocks to within 1 mV of the end value
v, n = 0.0, 0
while abs(v - VCM) > 1e-3:
    v = cycle(v, C_F, CR)[1]
    n += 1
print(f'start-up from 0 V, no load: {n} clocks to within 1 mV of vdd/2')

# the same with the sc_int2 load plates, which come from the ideal vg = vdd/2 and dump into vcm in phase 1
v, n_l, first = 0.0, 0, None
while abs(v - VCM) > 1e-3 and n_l < 200:
    v1, v = cycle(v, C_F, CR, 2 * C1_BENCH, VCM)
    first = v1 - VCM if first is None else first
    n_l += 1
print(f'start-up from 0 V with the bench load: {n_l} clocks, first sample {first*1e3:.0f} mV (sim: 12 clocks, first sample -273 mV, tt)')

# sensitivity to the virtual ground: vg 0.2 V above vdd/2, load plates 2*C1 on vcm in phase 1
def sens(c_units, cr_devs, c1):
    c = c_units * c_unit
    cr_ = cr_devs * 100 * GATE_DENS
    v = VCM
    for _ in range(4000):
        v1, v = cycle(v, c, cr_, 2 * c1, VCM + 0.2)
    v1b, vb = v1, v
    v = VCM
    for _ in range(4000):
        v1, v = cycle(v, c, cr_, 2 * c1, VCM)
    return (v1b - v1) / 0.2, (vb - v) / 0.2

for cu, crd, ref in ((8, 6, 0.27), (8, 12, 0.27), (16, 12, 0.15)):
    s1, s2 = sens(cu, crd, C1_BENCH)
    print(f'{cu:2d} flying units, {crd:2d} CR devices: end of phase 1 {s1:.3f}, end of phase 2 {s2:.3f} per volt of virtual ground (sim, end of phase 1: {ref})')

contraction per clock 0.6816, Rout from the recursion 0.96 MOhm, 1/(4 fs C) = 0.96 MOhm
start-up from 0 V, no load: 18 clocks to within 1 mV of vdd/2
start-up from 0 V with the bench load: 16 clocks, first sample -593 mV (sim: 12 clocks, first sample -273 mV, tt)
 8 flying units,  6 CR devices: end of phase 1 0.122, end of phase 2 0.098 per volt of virtual ground (sim, end of phase 1: 0.27)
 8 flying units, 12 CR devices: end of phase 1 0.117, end of phase 2 0.104 per volt of virtual ground (sim, end of phase 1: 0.27)
16 flying units, 12 CR devices: end of phase 1 0.065, end of phase 2 0.052 per volt of virtual ground (sim, end of phase 1: 0.15)


The recursion reproduces `1 / (4 fs C)` to 1 %, so the formula is right for the drawn pump. Start-up to 1 mV takes 18 clocks without the load and 16 with it, against 12 in the bench. The first sample of the bench (-273 mV) is not the first clock of the pump, so it does not compare with the model. It does not reproduce the sensitivity to the virtual ground either. The recursion gives about 0.1 per volt where the README measured 0.27 for 8 flying units and 0.15 for 16, a factor of 2 to 3 that is not explained here. The benches of that table used `cap_cmomi` flying capacitors with a substrate shunt, which this model does not have.

### Reservoir from the ripple

Each time the DAC of `sc_int2` changes level, the C1 right plates pull `C1 * Vdd` out of or into `vcm` at the start of phase 1. The flying capacitor has little say, the reservoir takes it, so `ripple = C1 * Vdd / CR` for one plate. The two plates see opposite DAC steps in the worst case, which doubles the span. The README measured 50.8 mV peak to peak at the end of phase 1 for CR = 2.7 pF.

The benches used C1 = 63.56 fF per plate. The `sc_int2` schematic carries 87.95 fF now, and 64 fF is the estimated real value. The table shows what each does to the reservoir that is drawn.

In [6]:
print(f'CR needed for {V_RIPPLE*1e3:.0f} mV, one plate: ', end='')
for name, c1 in (('bench 63.56 fF', C1_BENCH), ('real 64 fF', C1_REAL), ('model 87.95 fF', C1_MODEL)):
    print(f'{name} {c1*VDD/V_RIPPLE*1e12:.2f} pF, ', end='')
print()
for name, c1 in (('bench 63.56 fF', C1_BENCH), ('real 64 fF', C1_REAL), ('model 87.95 fF', C1_MODEL)):
    r = c1 * VDD / CR
    print(f'drawn CR {CR*1e12:.2f} pF with C1 {name:15s}: ripple {r*1e3:5.1f} mV one plate, {2*r*1e3:5.1f} mV for two (sim with the bench C1: 50.8 mV peak to peak)')

CR needed for 50 mV, one plate: bench 63.56 fF 1.91 pF, real 64 fF 1.92 pF, model 87.95 fF 2.64 pF, 
drawn CR 2.66 pF with C1 bench 63.56 fF : ripple  35.8 mV one plate,  71.7 mV for two (sim with the bench C1: 50.8 mV peak to peak)
drawn CR 2.66 pF with C1 real 64 fF     : ripple  36.1 mV one plate,  72.2 mV for two (sim with the bench C1: 50.8 mV peak to peak)
drawn CR 2.66 pF with C1 model 87.95 fF : ripple  49.6 mV one plate,  99.2 mV for two (sim with the bench C1: 50.8 mV peak to peak)


### Switches: Ron against the phase

The flying capacitor charges through `MS1` and `MS2` in phase 1 and discharges through `MS3` and `MS4` in phase 2. Each is a single device, so Ron is `1 / GDS` of the table at a small VDS and the gate drive the device has when its far terminal has settled: `MS1` and `MS4` at the rail with the full 1.5 V, `MS2` and `MS3` at `vcm` with 0.75 V and 0.75 V of body bias. The thick-oxide tables start at L = 0.5 um and the drawn `MS1` and `MS4` have 0.45 um, so those two are evaluated at 0.5 um, which is a little pessimistic. The sweep end of the tables (`VSB` 1.2 V) is not reached.

In [7]:
nmos  = lk(str(DATA / 'sg13cmos5l_lv_nmos.mat'))
pmos  = lk(str(DATA / 'sg13cmos5l_lv_pmos.mat'))
nmosh = lk(str(DATA / 'sg13cmos5l_hv_nmos.mat'))
pmosh = lk(str(DATA / 'sg13cmos5l_hv_pmos.mat'))
VDS = 0.01

def ron(tab, w, l, vgs, vsb):
    return 1 / (float(tab.lookup('GDS', L=l, VGS=vgs, VDS=VDS, VSB=vsb)) * w / tab['W'])

lh = max(DEV['MS1']['l'], 0.5)
R = {
    'MS1': ron(pmosh, DEV['MS1']['w'], lh, VDD, 0.0),
    'MS2': ron(pmos, DEV['MS2']['w'], DEV['MS2']['l'], VCM, VDD - VCM),
    'MS3': ron(nmos, DEV['MS3']['w'], DEV['MS3']['l'], VDD - VCM, VCM),
    'MS4': ron(nmosh, DEV['MS4']['w'], max(DEV['MS4']['l'], 0.5), VDD, 0.0),
}
for k, v in R.items():
    print(f"{k} {DEV[k]['sym']:12s} {DEV[k]['w']:.0f}u / {DEV[k]['l']:.2f}u: Ron {v/1e3:6.2f} kohm")
R_MAX = TS2 / (N_SETTLE * C_F)
for ph, (a, b) in (('phase 1', ('MS1', 'MS2')), ('phase 2', ('MS3', 'MS4'))):
    r = R[a] + R[b]
    print(f'{ph}: {r/1e3:.1f} kohm x {C_F*1e15:.0f} fF = {r*C_F*1e9:.2f} ns, {TS2/(r*C_F):.0f} time constants in {TS2*1e9:.0f} ns, allowed {R_MAX/1e3:.0f} kohm for {N_SETTLE}')
print(f"gate area MS2 {DEV['MS2']['w']*DEV['MS2']['l']:.2f} um2, MS3 {DEV['MS3']['w']*DEV['MS3']['l']:.2f} um2 (equal, so their injection into vcm cancels)")

MS1 sg13_hv_pmos 4u / 0.45u: Ron   3.21 kohm
MS2 sg13_lv_pmos 4u / 0.50u: Ron   6.08 kohm
MS3 sg13_lv_nmos 4u / 0.50u: Ron   0.94 kohm
MS4 sg13_hv_nmos 2u / 0.45u: Ron   1.73 kohm
phase 1: 9.3 kohm x 254 fF = 2.36 ns, 207 time constants in 488 ns, allowed 240 kohm for 8
phase 2: 2.7 kohm x 254 fF = 0.68 ns, 720 time constants in 488 ns, allowed 240 kohm for 8
gate area MS2 2.00 um2, MS3 2.00 um2 (equal, so their injection into vcm cancels)


Ron is a few kohm and the allowed value is 240 kohm, so the switch widths are not set by settling. They are set by leakage and injection. The README gives the leak of the old transmission gate as the reason for the single devices: with Vgs = 0 and 0.78 V across, the NMOS drew 62 nA at ff 125 C. The tables are tt 300 K and give the same device at its tt value.

In [8]:
def i_off(tab, w, l, vds):
    return float(tab.lookup('ID', L=l, VGS=0.0, VDS=vds, VSB=0.0)) * w / tab['W']

i_old = i_off(nmos, DEV['MS3']['w'], DEV['MS3']['l'], 0.78)
i_hv = i_off(nmosh, DEV['MS4']['w'], max(DEV['MS4']['l'], 0.5), VCM)
print(f'old tgate NMOS, Vgs = 0, 0.78 V across, {DEV["MS3"]["w"]:.0f}u/{DEV["MS3"]["l"]:.1f}u at tt: {i_old*1e9:.2f} nA, {i_old*R_OUT*1e3:.2f} mV on vcm (sim 62 nA at ff 125 C)')
print(f'thick-oxide NMOS {DEV["MS4"]["w"]:.0f}u/{max(DEV["MS4"]["l"],0.5):.1f}u at Vgs = 0, 0.75 V across at tt: {i_hv*1e9:.1e} nA')
print(f'the drawn single devices sit at Vgs = {-VCM} V when off, the table does not go there')

old tgate NMOS, Vgs = 0, 0.78 V across, 4u/0.5u at tt: 1.37 nA, 1.32 mV on vcm (sim 62 nA at ff 125 C)
thick-oxide NMOS 2u/0.5u at Vgs = 0, 0.75 V across at tt: 2.4e-05 nA
the drawn single devices sit at Vgs = -0.75 V when off, the table does not go there


### Decoupling

`MDEC` is the `vdd` decoupling in the layout corners, gate on `vdd`.

In [9]:
c_dec = DEV['MDEC']['m'] * DEV['MDEC']['w'] * DEV['MDEC']['l'] * GATE_DENS
print(f'MDEC {DEV["MDEC"]["m"]} x {DEV["MDEC"]["w"]:.0f}u x {DEV["MDEC"]["l"]:.0f}u: {c_dec*1e12:.2f} pF at {GATE_DENS*1e15:.1f} fF/um2 (README: about 0.3 pF)')

MDEC 2 x 4u x 10u: 0.30 pF at 3.8 fF/um2 (README: about 0.3 pF)


### Derived against drawn

In [10]:
r1 = R['MS1'] + R['MS2']
r2 = R['MS3'] + R['MS4']
rows = [
    ('CF1 + CF2 (derived is the minimum)', C_MIN * 1e15, C_F * 1e15, 'fF', C_F >= C_MIN),
    ('Rout (derived is the maximum)', R_OUT_MAX / 1e6, R_OUT / 1e6, 'MOhm', R_OUT <= R_OUT_MAX),
    ('Rout, recursion against 1/(4 fs C)', R_OUT / 1e6, r_eff / 1e6, 'MOhm', abs(r_eff / R_OUT - 1) < 0.1),
    ('start-up to 1 mV with the bench load against sim', n_l, 12, 'clocks', abs(n - 12) <= 2),
    ('CR for 50 mV, bench C1 (derived is the minimum)', C1_BENCH * VDD / V_RIPPLE * 1e12, CR * 1e12, 'pF', CR >= C1_BENCH * VDD / V_RIPPLE),
    ('CR for 50 mV, sc_int2 model C1', C1_MODEL * VDD / V_RIPPLE * 1e12, CR * 1e12, 'pF', CR >= C1_MODEL * VDD / V_RIPPLE),
    ('ripple one plate, bench C1, against sim p-p', C1_BENCH * VDD / CR * 1e3, 50.8, 'mV', C1_BENCH * VDD / CR * 1e3 < 50.8 < 2 * C1_BENCH * VDD / CR * 1e3),
    ('Ron phase 1 allowed / MS1 + MS2', R_MAX / 1e3, r1 / 1e3, 'kohm', r1 < R_MAX),
    ('Ron phase 2 allowed / MS3 + MS4', R_MAX / 1e3, r2 / 1e3, 'kohm', r2 < R_MAX),
    ('MS2 / MS3 gate area', DEV['MS2']['w'] * DEV['MS2']['l'], DEV['MS3']['w'] * DEV['MS3']['l'], 'um2', DEV['MS2']['w'] * DEV['MS2']['l'] == DEV['MS3']['w'] * DEV['MS3']['l']),
    ('virtual ground sensitivity, 8 units CR 6, sim', sens(8, 6, C1_BENCH)[0], 0.27, '', abs(sens(8, 6, C1_BENCH)[0] / 0.27 - 1) < 0.25),
]
print(f'{"quantity":50s}{"derived":>10s}{"drawn/sim":>11s}  unit    agree')
for name, d, s, unit, ok in rows:
    print(f'{name:50s}{d:10.3g}{s:11.3g}  {unit:7s} {"yes" if ok else "NO"}')

quantity                                             derived  drawn/sim  unit    agree
CF1 + CF2 (derived is the minimum)                       160        254  fF      yes
Rout (derived is the maximum)                           1.53      0.961  MOhm    yes
Rout, recursion against 1/(4 fs C)                     0.961      0.958  MOhm    yes
start-up to 1 mV with the bench load against sim          16         12  clocks  NO
CR for 50 mV, bench C1 (derived is the minimum)         1.91       2.66  pF      yes
CR for 50 mV, sc_int2 model C1                          2.64       2.66  pF      yes
ripple one plate, bench C1, against sim p-p             35.8       50.8  mV      yes
Ron phase 1 allowed / MS1 + MS2                          240       9.29  kohm    yes
Ron phase 2 allowed / MS3 + MS4                          240       2.67  kohm    yes
MS2 / MS3 gate area                                        2          2  um2     yes
virtual ground sensitivity, 8 units CR 6, sim          0.122    

### Result

The drawn pump follows from the spec. `CF1 + CF2` is 254 fF against the 163 fF the assumed 5 mV allows, the resistance is 0.96 MOhm and the exact two-phase recursion confirms the `1 / (4 fs C)` formula. Ron is 9.3 kohm in phase 1 and 2.7 kohm in phase 2 against 240 kohm allowed, and `MS2` and `MS3` have equal gate area. Disagreements, none changed:

- The reservoir is sized for the bench C1 of 63.56 fF (ripple 35 mV for one plate, 50.8 mV peak to peak simulated). The `sc_int2` schematic now carries 87.95 fF per C1 (64 fF estimated real), which makes the ripple 1.38 times larger with the model value, 49 mV for one plate. The README sweeps and the closed loop run predate that capacitor.
- Start-up from 0 V to within 1 mV takes 16 clocks in the recursion with the load and 12 in the bench. The cause is open, the bench starts the pump at 2.2 clocks and may hold other initial charge.
- The sensitivity of `vcm` to the virtual grounds is 0.27 per volt in the README for 8 flying units and about 0.1 in the exact recursion. The README benches used `cap_cmomi` flying capacitors and the cause is open.
- `MS1` and `MS4` are drawn 0.45 um and the thick-oxide tables start at 0.5 um, so their Ron is evaluated at 0.5 um.
- The 5 mV error and the 50 mV ripple are assumptions of this notebook, the README states neither.
- The README's tgate leak (62 nA at ff 125 C) cannot be checked at tt 300 K. The table gives 1.4 nA at tt for the same device, which is a factor 44 for the corner, and the off state of the drawn single devices lies outside the table.